# 📖 第五课：决策树基础

**目标**：理解决策树的构建原理，掌握信息熵与信息增益，学会用决策树进行分类

---

## 📚 学习目标
- 理解决策树的直观结构（根节点、内部节点、叶节点）
- 掌握信息熵、信息增益、基尼不纯度的计算
- 学会使用 sklearn 的 DecisionTreeClassifier
- 理解决策树深度与过拟合的关系
- 了解决策树的可解释性优势

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.font_manager as fm
import pathlib
_cache = matplotlib.get_cachedir()
for _f in pathlib.Path(_cache).glob('fontlist*.json'):
    _f.unlink(missing_ok=True)
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.sans-serif'] = ['PingFang HK', 'PingFang SC', 'Heiti TC', 'Heiti SC', 'STHeiti', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("库导入成功！")

---

## 1. 信息熵 (Information Entropy)

熵衡量数据的**不确定性**，熵越大越混乱：

$$H = -\sum p_i \log_2(p_i)$$

In [ ]:
# 手动实现信息熵
def entropy(y):
    """计算信息熵"""
    counts = np.bincount(y)
    probs = counts / len(y)
    probs = probs[probs > 0]  # 去掉概率为0的
    return -np.sum(probs * np.log2(probs))

# 直觉理解：不同纯度的数据
print("信息熵示例:")
print("="*40)

# 完全纯净（全是一个类）
y_pure = np.array([0, 0, 0, 0, 0])
print(f"纯数据 [0,0,0,0,0]: 熵={entropy(y_pure):.3f}")

# 完全混合
y_mixed = np.array([0, 1, 0, 1, 0])
print(f"混合数据 [0,1,0,1,0]: 熵={entropy(y_mixed):.3f}")

# 完全随机
y_random = np.array([0, 1, 0, 1, 0, 1])
print(f"均匀分布 [0,1,0,1,0,1]: 熵={entropy(y_random):.3f}")

# 可视化熵随概率变化的曲线
p = np.linspace(0.01, 0.99, 100)
h = -p * np.log2(p) - (1-p) * np.log2(1-p)

plt.figure(figsize=(8, 5))
plt.plot(p, h, 'b-', linewidth=2)
plt.xlabel('正类概率 p')
plt.ylabel('信息熵 H')
plt.title('二元信息熵曲线')
plt.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='p=0.5 (最混乱)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n💡 熵越大 → 数据越混乱 → 越需要分割")

---

## 2. 信息增益 (Information Gain)

信息增益 = 分割前的熵 - 分割后的加权熵

决策树每次选择**信息增益最大的特征和阈值**进行分割。

In [ ]:
def information_gain(X, y, feature_idx, threshold):
    """计算按某特征某阈值分割后的信息增益"""
    # 分割前的熵
    h_parent = entropy(y)
    
    # 分割数据
    left_mask = X[:, feature_idx] <= threshold
    right_mask = ~left_mask
    
    if left_mask.sum() == 0 or right_mask.sum() == 0:
        return 0  # 无效分割
    
    # 分割后的加权熵
    n = len(y)
    h_left = entropy(y[left_mask])
    h_right = entropy(y[right_mask])
    h_children = (left_mask.sum()/n) * h_left + (right_mask.sum()/n) * h_right
    
    return h_parent - h_children

# 用一个简单数据集演示
np.random.seed(42)
X_demo = np.array([[1, 2], [2, 3], [3, 1], [4, 5], [5, 4], [6, 6]])
y_demo = np.array([0, 0, 0, 1, 1, 1])

print("演示数据:")
for i in range(len(X_demo)):
    print(f"  特征{X_demo[i]} → 类别 {y_demo[i]}")

print(f"\n初始熵: {entropy(y_demo):.3f}")

# 计算每个特征每个阈值的信息增益
print("\n各分割点的信息增益:")
for feat in range(2):
    for thresh in np.unique(X_demo[:, feat])[:-1]:
        ig = information_gain(X_demo, y_demo, feat, thresh)
        print(f"  特征{feat} ≤ {thresh}: 信息增益 = {ig:.3f}")

---

## 3. 使用 sklearn 构建决策树

In [ ]:
# 生成分类数据
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                           n_informative=2, random_state=42, n_clusters_per_class=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 训练决策树
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

# 评估
y_pred = tree.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"决策树准确率: {accuracy:.3f}")
print(f"树深度: {tree.get_depth()}")
print(f"叶节点数: {tree.get_n_leaves()}")

In [ ]:
# 可视化决策树结构
plt.figure(figsize=(16, 8))
plot_tree(tree, filled=True, feature_names=['特征1', '特征2'],
          class_names=['类别A', '类别B'], fontsize=10)
plt.title('决策树结构 (max_depth=3)')
plt.show()

print("\n💡 每个节点显示:")
print("  - 分割条件 (特征 ≤ 阈值)")
print("  - 基尼不纯度 (越小越纯)")
print("  - 样本数")
print("  - 各类别数量")
print("  - 多数类别")

In [ ]:
# 可视化决策边界
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 决策边界绘制辅助函数
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

for ax, depth in zip(axes, [1, 3, 10]):
    t = DecisionTreeClassifier(max_depth=depth, random_state=42)
    t.fit(X_train, y_train)
    
    Z = t.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='viridis',
              edgecolors='black', s=30, alpha=0.7)
    
    train_acc = accuracy_score(y_train, t.predict(X_train))
    test_acc = accuracy_score(y_test, t.predict(X_test))
    ax.set_title(f'depth={depth}\n训练: {train_acc:.3f}, 测试: {test_acc:.3f}')
    ax.set_xlabel('特征1')
    ax.set_ylabel('特征2')

plt.suptitle('不同深度的决策边界', fontsize=14)
plt.tight_layout()
plt.show()

print("\n💡 深度越大 → 边界越复杂 → 可能过拟合")

---

## 4. 基尼不纯度 vs 信息熵

sklearn 默认使用**基尼不纯度** (Gini Impurity) 作为分割标准：

$$Gini = 1 - \sum p_i^2$$

- 计算比信息熵更快
- 结果通常非常接近

In [ ]:
# 对比两种分割标准
tree_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
tree_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)

# 交叉验证对比
gini_scores = cross_val_score(tree_gini, X, y, cv=5)
entropy_scores = cross_val_score(tree_entropy, X, y, cv=5)

print("分割标准对比 (5折交叉验证):")
print(f"  Gini:   {gini_scores.mean():.3f} ± {gini_scores.std():.3f}")
print(f"  Entropy: {entropy_scores.mean():.3f} ± {entropy_scores.std():.3f}")
print("\n💡 两种标准结果通常很接近，Gini 计算更快")

---

## 5. 决策树深度与过拟合

In [ ]:
# 不同深度下的训练/测试准确率
depths = range(1, 20)
train_accs = []
test_accs = []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, t.predict(X_train)))
    test_accs.append(accuracy_score(y_test, t.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(depths, train_accs, 'b-o', label='训练准确率', markersize=5)
plt.plot(depths, test_accs, 'r-s', label='测试准确率', markersize=5)
plt.xlabel('树深度')
plt.ylabel('准确率')
plt.title('决策树深度 vs 准确率')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

best_depth = depths[np.argmax(test_accs)]
print(f"最佳深度: {best_depth} (测试准确率: {max(test_accs):.3f})")
print("\n💡 决策树太深 → 记住训练数据 → 过拟合")

---

## 6. 综合实战：鸢尾花分类

In [ ]:
from sklearn.datasets import load_iris

# 加载鸢尾花数据集
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)

# 训练决策树
tree_iris = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_iris.fit(X_tr, y_tr)

y_pred = tree_iris.predict(X_te)
print(f"鸢尾花分类准确率: {accuracy_score(y_te, y_pred):.3f}")
print("\n分类报告:")
print(classification_report(y_te, y_pred, target_names=iris.target_names))

# 可视化特征重要性
plt.figure(figsize=(8, 5))
plt.bar(iris.feature_names, tree_iris.feature_importances_, color='steelblue')
plt.title('决策树特征重要性')
plt.ylabel('重要性')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n💡 特征重要性：衡量每个特征对决策的贡献程度")

---

## 📝 课后思考

1. 决策树为什么容易过拟合？如何防止？
2. 如果一个特征有缺失值，决策树怎么处理？
3. 为什么说决策树是「可解释」的模型？这对实际应用有什么意义？

---

**下一步**：随机森林与集成学习——将多棵决策树组合成更强大的模型，理解 bagging 和 boosting 的思想。